In [1]:
%pip install osmnx networkx geopandas shapely folium earthengine-api pandas numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import ee

ee.Authenticate()
ee.Initialize(project='project-0cf410a3-f35f-4911-9c5')

PLACE_NAME = "Tarangire National Park, Tanzania"

In [ ]:
from park_road_network import ParkRoadNetwork
from chirps_provider import CHIRPSProvider
from edge_risk_enricher import EdgeRiskEnricher


nodes, edges = ParkRoadNetwork(PLACE_NAME).load()
    rain_provider = CHIRPSProvider()

    pixel_grid = rain_provider.frame_grid(edges.total_bounds)

    rain_pixels, image_collection = rain_provider.get_accumulated_precipitation(
        pixel_grid,
        start_date=start_date.strftime("%Y-%m-%d"), #TODO: dynamic selection and avoid str as input
        end_date=end_date.strftime("%Y-%m-%d")
    )

    enricher = EdgeRiskEnricher()
    edges_enriched = enricher.enrich(edges, rain_pixels)

    center_lat, center_lon = nodes["y"].mean(), nodes["x"].mean()
    m = folium.Map(location=[center_lat, center_lon], zoom_start=10)

    SURFACE_RISK_COLORS = {
        (0.0, 0.25): "#2ecc71",
        (0.25, 0.5): "#f39c12",
        (0.5, 0.75): "#e74c3c",
        (0.75, 1.01): "#7b241c",
    }